# LEGO Supply-Chain & Sustainability Knowledge Graph (Prototype)

> **IMPORTANT: Case Project & Demonstration Disclaimer**
> This repository is a **Strategic Case Project** created for demonstration purposes, specifically for the **Senior Product Manager - Enterprise Knowledge Platform** role at the **LEGO Group**. 
>
> **Methodology**: This platform is a **high-leverage synergy** between advanced LLM tools (Gemini CLI) and deep multi-disciplinary expertise (Product, SCM, Agile, Leadership). It demonstrates high-fidelity logic and domain depth at an accelerated pace.
>
> **Product Maturity: 30% (Strategic MVP)**
> *100% represents a production-ready, enterprise-federated platform (SAP/JDA/Workday integration).*

## 1. Project Overview
This notebook demonstrates a prototype Enterprise Knowledge Platform (EKP) for the LEGO Group, focusing on supply-chain resilience and sustainability. 

### Personas Served:
- **Supply Chain Planner**: Identifying single points of failure and production dependencies.
- **Sustainability Analyst**: Mapping ESG risk from suppliers to finished sets.
- **Platform Engineer**: Ensuring data is FAIR(EST) (Findable, Accessible, Interoperable, Reusable).

### How to use this notebook (The Guided Tour)
1. **Setup**: The first few cells load the environment and connection to Neo4j.
2. **Run All**: You can safely click *'Run All'* to execute the data pipeline from top to bottom.
3. **Explore**: Each section answers a specific business question with a short summary ('Why this matters') followed by the data.

## 1.1 Product Strategy: The 4 Big Risks
As the Senior PM for the EKP, every scenario in this prototype is selected to mitigate the **4 Big Risks** of product discovery (Cagan Framework), ensuring that the Knowledge Platform is not just technically feasible, but business-viable.

| Risk Category | EKP Mitigation Strategy |
| :--- | :--- |
| **Value Risk** | Will users (Planners/Analysts) choose to use it? We solve this by answering "unanswerable" cross-functional questions (e.g., Tier 2 SPOFs). |
| **Usability Risk** | Can users figure it out? We implement a "Guided Tour" UX and message-driven visualizations. |
| **Feasibility Risk** | Can we build it? We use **Neo4j** and **FAIR(EST) principles** to prove that complex relationship modeling is technically achievable today. |
| **Business Viability Risk** | Does it work for LEGO? We align semantics with **SCOR DS v14.0** and **2032 Sustainability Targets** to ensure the solution fits the LEGO Group's operational and strategic core. |

> **Strategic Assumption**: Value Chain management is the bedrock of the LEGO Group. Most enterprise risks and opportunities lie in the "connective tissue" between suppliers, factories, and themes.

### 1.2 The EKP Stakeholder JTBD Catalog
Beyond the Primary Personas (Planners/Engineers), the EKP serves key "Jobs to be Done" (JTBD) for LEGO's secondary stakeholders (the "Sawdust" use cases).

| Stakeholder | The "Job" (JTBD Statement) |
| :--- | :--- |
| **Marketing** | "When preparing a campaign, I want to verify the mass-balance of every supplier in that theme, so that we can avoid greenwashing risk." |
| **Finance** | "When setting 2027 prices, I want to see Tier 2 material exposure, so that I can build a data-backed margin forecast." |
| **PR / Comms** | "When a crisis hits the news, I want to immediately identify which themes are *unaffected*, so that I can reassure our retail partners." |

### 1.3 Co-Piloted by Gemini CLI: The 4D Framework
This project is an example of **High-Leverage AI Collaboration**. We utilize the **4D Framework** to demonstrate responsible and transparent development:

| Principle | Collaboration Strategy |
| :--- | :--- |
| **Delegation** | Strategic vision (Alexander) vs. Autonomous engineering (Gemini CLI). |
| **Description** | Effective communication via structured workflows and JTBD prompts. |
| **Discernment** | Critical evaluation through **Domain Audits** and **Data Confidence** flagging. |
| **Diligence** | Responsible AI use, explicitly marking synthetic data to avoid misinformation. |

> **Disclaimer**: All code in this notebook was generated by Gemini CLI and verified by the Human Strategic Lead. Synthetic data used for "Platform Potential" scenarios is explicitly labeled with `is_synthetic: true` to ensure transparency and prevent misinformation.

## 2. FAIR(EST) Data Inventory
A key component of an EKP is the transparent governance of data sources. 

| Asset | Source | Type | FAIR(EST) Alignment | Status |
| :--- | :--- | :--- | :--- | :--- |
| **Supplier List** | LEGO 2024 PDF | Real | Standardized names/locations | Loaded |
| **Factory List** | Official LEGO News | Real | Verified geographic locations | Loaded |
| **Sets & Themes** | Rebrickable | Real | Stable IDs (Interoperable) | Loaded (Star Wars) |
| **Inbound Links** | Synthetic | Simulated | `is_synthetic: true` (Governance) | Wired |
| **Production Links** | Synthetic | Simulated | `is_synthetic: true` (Governance) | Wired |

In [ ]:
import os
from neo4j import GraphDatabase
from dotenv import load_dotenv
import pandas as pd
from pyvis.network import Network
from IPython.display import IFrame

load_dotenv("../.env.local")

NEO4J_URI = os.getenv("NEO4J_INSTANCE_CONNECTION_URI")
NEO4J_USER = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_query(query):
    with driver.session() as session:
        result = session.run(query)
        return pd.DataFrame([dict(record) for record in result])

def visualize_graph(query, output_filename="graph.html"):
    net = Network(height="500px", width="100%", bgcolor="#222222", font_color="white", notebook=True, cdn_resources='in_line')
    with driver.session() as session:
        results = session.run(query)
        for record in results:
            for key in record.keys():
                item = record[key]
                if hasattr(item, 'nodes'): # It's a Path
                    for node in item.nodes:
                        label = list(node.labels)[0]
                        name = node.get('name') or node.get('set_num') or str(node.id)
                        color = "#97c2fc"
                        if label == 'Supplier': color = "#ff9999"
                        elif label == 'Factory': color = "#ffff99"
                        elif label == 'Set': color = "#99ff99"
                        elif label == 'Theme': color = "#c2f0c2"
                        net.add_node(node.element_id, label=f"{label}: {name}", title=str(dict(node)), color=color)
                    for rel in item.relationships:
                        net.add_edge(rel.start_node.element_id, rel.end_node.element_id, title=rel.type)
    net.show(output_filename)
    return IFrame(src=output_filename, width="100%", height="550px")

# 1. Graph Statistics
stats_query = """
MATCH (n) 
RETURN labels(n)[0] AS Label, count(*) AS Count
UNION ALL
MATCH ()-[r]->()
RETURN type(r) AS Label, count(*) AS Count
"""
stats_df = run_query(stats_query)
print("--- Graph Statistics ---")
print(stats_df)

## 2.1 Platform Health & FAIR(EST) Metrics
As a Senior PM, data trust is paramount. This query monitors the ratio of real vs. synthetic data and highlights nodes with low confidence scores.

In [ ]:
# Platform Health Dashboard
health_check_query = """
MATCH (n)
RETURN 
    count(n) AS Total_Nodes,
    sum(CASE WHEN n.is_synthetic = true THEN 1 ELSE 0 END) AS Synthetic_Nodes,
    sum(CASE WHEN n.dataConfidence < 0.5 THEN 1 ELSE 0 END) AS Low_Confidence_Nodes
"""
try:
    health_df = run_query(health_check_query)
    print("--- Platform Health Dashboard ---")
    print(health_df)
except NameError:
    pass

### 2.2 FY2026 Platform OKR Tracker (Simulation)
As the Senior PM, I track the EKP's progress through **Objectives and Key Results (OKRs)**, ensuring technical work is tied to business outcomes.

| Objective | Key Result | Progress (Current) |
| :--- | :--- | :--- |
| **O1: Achieve Enterprise-Scale Data Trust** | KR1: 95% of Tier 1/2 nodes meet "FAIR(EST)-Gold" standards. | 🟢 100% (Metadata present) |
| **O2: Empower Proactive Resilience** | KR2: Support 10+ "What-If" disruption scenarios. | 🟡 1 (Logistics Event) |
| **O3: Drive Cross-Functional Adoption** | KR3: Onboard 50+ users from 3+ departments. | ⚪ Pilot Phase |

In [ ]:
# 2. Sample Supply Chain Paths
path_query = """
MATCH (s:Supplier)-[:SOURCES_TO]->(f:Factory)-[:TRANSFORMS_TO]->(set:Set)
RETURN s.name AS Supplier, f.name AS Factory, set.name AS LEGO_Set
LIMIT 10
"""
path_df = run_query(path_query)
print("\n--- Sample Supply Chain Paths (Synthetic) ---")
print(path_df)

## 3. Decision Support: Supply Chain Resilience (T-06)

Identifying **Single Points of Failure (SPOF)** is critical for proactive risk management. If a set is built by only one factory, or a factory is supplied by only one supplier, the entire production line is vulnerable.\n\n**Hypothesis**: By modeling the 1:N relationship between Suppliers and Factories, we will uncover "Hidden SPOFs" that are invisible to 80% of S&OP planners.

**Success Metric**: Number of critical theme-factory dependencies identified per query.

**Strategic Choice**: This scenario targets **Business Viability Risk**. By identifying SPOFs, we protect the core LEGO business model from high-impact disruptions.

**Executive Summary**: Identifying Single Points of Failure (SPOF) allows us to move from 'Firefighting' to 'Strategic Sourcing'. Each row below represents a production line that can be hardened through diversification.

In [ ]:
# 1. Critical Sets: Produced at exactly ONE Factory
spof_sets_query = """
MATCH (f:Factory)-[:TRANSFORMS_TO]->(s:Set)
WITH s, count(f) AS factory_count, collect(f.name) AS factories
WHERE factory_count = 1
RETURN s.name AS Set_Name, s.set_num AS Set_Num, factories[0] AS Sole_Factory
ORDER BY Set_Name
LIMIT 10
"""
spof_sets_df = run_query(spof_sets_query)
print("--- Vulnerable Sets (Single Factory) ---")
print(spof_sets_df)

In [ ]:
# 2. Critical Factories: Supplied by exactly ONE Supplier
spof_factories_query = """
MATCH (sup:Supplier)-[:SOURCES_TO]->(f:Factory)
WITH f, count(sup) AS supplier_count, collect(sup.name) AS suppliers
WHERE supplier_count = 1
RETURN f.name AS Factory_Name, f.location AS Location, suppliers[0] AS Sole_Supplier
"""
spof_factories_df = run_query(spof_factories_query)
print("\n--- Vulnerable Factories (Single Supplier) ---")
print(spof_factories_df)

In [ ]:
# 3. High Risk Propagation (SPOF + Supplier Risk Score)
risk_query = """
MATCH (s:Set)
WHERE count { (:Factory)-[:TRANSFORMS_TO]->(s) } = 1
MATCH (sup:Supplier)-[:SOURCES_TO]->(f:Factory)-[:TRANSFORMS_TO]->(s)
WHERE count { (:Supplier)-[:SOURCES_TO]->(f) } = 1
RETURN s.name AS Set_Name, f.name AS Factory, sup.name AS Supplier, sup.risk_score AS Supplier_Risk, sup.risk_factor AS Risk_Factor
ORDER BY Supplier_Risk DESC
LIMIT 10
"""
risk_df = run_query(risk_query)
print("\n--- High-Risk Vulnerable Production Lines ---")
print(risk_df)


## 4. Strategic Dashboard: 2032 Sustainability Targets (T-12)

Measuring the **Distance to 2032 Target (DtT)** across the supply chain. LEGO's goal is **100% Sustainable Materials** by 2032. This dashboard transforms technical risk scores into strategic progress metrics.

**Key Metrics:**
- **Sustainability Completion % (SC%)**: Progress towards the 2032 goal (0-100%).
- **Distance to 2032 Target (DtT)**: The remaining gap ($100 - SC\%$).
- **Material Mix**: Ratio of Bio-PE/Recycled vs. Petro-based ABS.

**Strategic Choice**: This scenario targets **Value Risk**. It provides the Sustainability team with the unique "Product-Level Traceability" they need to hit 2032 milestones and report progress to stakeholders.

In [ ]:
# 1. Material Mix: Sustainable vs. Conventional Components
material_mix_query = """
MATCH (m:Material)
RETURN m.name AS Material, 
       m.sustainability_completion_pct AS SC_Percentage, 
       m.distance_to_2032_target AS Distance_to_Target
ORDER BY SC_Percentage DESC
"""
material_mix_df = run_query(material_mix_query)
print("--- Portfolio Material Mix: 2032 Baseline ---")
print(material_mix_df)

In [ ]:
# 2. Executive View: Themes with Highest 'Distance to 2032 Target'
theme_dtt_query = """
MATCH (t:Theme)
WHERE t.distance_to_2032_target IS NOT NULL
RETURN t.name AS Theme, 
       round(t.sustainability_completion_pct) AS Completion_Pct, 
       round(t.distance_to_2032_target) AS Gap_to_2032
ORDER BY Gap_to_2032 DESC
LIMIT 10
"""
theme_dtt_df = run_query(theme_dtt_query)
print("\n--- Sustainability 'Gap Analysis' by Theme ---")
print(theme_dtt_df)

In [ ]:
# 2.1 Visualization: Distance to 2032 Target by Theme
import matplotlib.pyplot as plt
if not theme_dtt_df.empty:
    theme_dtt_df.plot(kind='bar', x='Theme', y='Gap_to_2032', color='orange', title='Strategic Gap: Distance to 2032 ESG Target')
    plt.ylabel('Percentage Points Remaining')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

In [ ]:
# 3. Innovation Spotlight: Botanicals vs. Technic (Material Dependency)
innovation_query = """
MATCH (t:Theme)-[r:DEPENDS_ON_MATERIAL]->(m:Material)
WHERE t.name IN ['Botanicals', 'Technic', 'Creator Expert']
RETURN t.name AS Theme, 
       m.name AS Primary_Material, 
       round(t.sustainability_completion_pct) AS Final_SC_Pct, 
       round(t.offset_impact) AS Carbon_Offsets_Applied
"""
innovation_df = run_query(innovation_query)
print("\n--- Theme Innovation Spotlight: Material Dependency & Offsets ---")
print(innovation_df)

## 5. 'Sawdust' Use Cases: Marketing & PR
**Why this matters**: A Knowledge Graph serves multiple stakeholders. Here, we query the graph for the Marketing team to identify Themes that have achieved >80% Mass Balance certification, ready for a sustainability press release.

In [ ]:
# PR Milestone: Highly Certified Themes
pr_query = """
MATCH (t:Theme)<-[:IN_THEME]-(s:Set)<-[:TRANSFORMS_TO]-(f:Factory)<-[:SOURCES_TO]-(sup:Supplier)
WITH t, count(sup) AS total_suppliers, sum(CASE WHEN sup.mass_balance_certified = true THEN 1 ELSE 0 END) AS certified_suppliers
WITH t, (toFloat(certified_suppliers) / total_suppliers) * 100 AS Cert_Percentage
WHERE Cert_Percentage > 80
RETURN t.name AS Theme, round(Cert_Percentage) AS Certification_Percentage
ORDER BY Cert_Percentage DESC
"""
try:
    pr_df = run_query(pr_query)
    print("--- PR Ready Themes (>80% Sustainable Materials) ---")
    print(pr_df)
except NameError:
    pass

## 6. Proactive Modeling: Macro-Economic Disruptions
**Why this matters**: The graph isn't just reactive. By injecting a temporary `RiskEvent` (e.g., 'Red Sea Shipping Delay'), we can proactively forecast which suppliers, and therefore which finished goods, will be disrupted in the next 6-12 months.

In [ ]:
# Injecting a Macro-Economic Risk Event
mock_risk_query = """
// 1. Create a mock global event
MERGE (r:RiskEvent {name: 'Red Sea Shipping Delay', type: 'Logistics Disruption', severity: 'High'})
WITH r
// 2. Link it to Asian/Middle Eastern Suppliers
MATCH (sup:Supplier)
WHERE sup.location IN ['China', 'Vietnam', 'Taiwan', 'Malaysia', 'India']
MERGE (r)-[:AFFECTS]->(sup)
WITH r
// 3. Query downstream impact
MATCH (r)-[:AFFECTS]->(sup:Supplier)-[:SOURCES_TO]->(f:Factory)-[:TRANSFORMS_TO]->(s:Set)-[:IN_THEME]->(t:Theme)
RETURN r.name AS Active_Risk, sup.name AS Disrupted_Supplier, count(DISTINCT s) AS Sets_At_Risk, collect(DISTINCT t.name)[0..3] AS Affected_Themes
LIMIT 5
"""
try:
    risk_forecast_df = run_query(mock_risk_query)
    print("--- Proactive Disruption Forecast ---")
    print(risk_forecast_df)
except NameError:
    pass

## 7. Interactive Knowledge Graph Visualization (T-08)

Visualizing complex supply-chain dependencies to pinpoint vulnerabilities and ESG hotspots.

In [ ]:
# 1. Visualizing Vulnerability Path for a specific Set
vulnerability_path_query = """
MATCH path = (sup:Supplier)-[:SOURCES_TO]->(f:Factory)-[:TRANSFORMS_TO]->(s:Set {name: 'Millennium Falcon'})
RETURN path
"""
visualize_graph(vulnerability_path_query, "vulnerability_path.html")

In [ ]:
# 2. Sustainability Hotspots: Mapping High-Risk Suppliers to Themes
hotspots_query = """
MATCH path = (sup:Supplier)-[:SOURCES_TO]->(f:Factory)-[:TRANSFORMS_TO]->(s:Set)-[:IN_THEME]->(t:Theme)
WHERE sup.esg_risk_score > 0.7
RETURN path
LIMIT 5
"""
visualize_graph(hotspots_query, "sustainability_hotspots.html")

## 8. Multi-Tier (Tier 2) Risk Propagation
Visualizing hidden Single Points of Failure originating from Tier 2 material suppliers.

In [ ]:
# Visualizing Hidden Tier 2 Risk Propagation
t2_query = """
MATCH path = (t2:Supplier {tier: 2})-[:SOURCES_TO]->(t1:Supplier)-[:SOURCES_TO]->(f:Factory)-[:TRANSFORMS_TO]->(s:Set)
WHERE t2.risk_score > 0.8
RETURN path
LIMIT 10
"""
visualize_graph(t2_query, "t2_risk_path.html")

## 9. Strategic Roadmap: EKP v1.0 - v3.0 (30% Progress)

This prototype represents **Phase 1 (Ingestion & Strategic Baseline)** of a larger Enterprise Knowledge Platform strategy. We are currently at **30% Maturity (Strategic MVP)**.

| Phase | Milestone | Strategic Outcome |
| :--- | :--- | :--- |
| **Phase 1 (30%)** | **Architecture & Multi-Tier Logic** | DtT Metrics, Ripple Effect simulation, and SCOR DS alignment. |
| **Phase 2 (60%)** | **Knowledge Federation** | Bridging internal SAP (Materials) and JDA (Logistics) data into a single Federated Graph (Neo4j Fabric). |
| **Phase 3 (90%)** | **Cognitive Logic Layer** | Integrating LLMs for natural language querying ("Which themes are at risk of a 10% price hike?"). |
| **Phase 4 (100%)** | **Autonomous Resilience** | Automated supply-chain rerouting and real-time circular economy tracking. |

### Conclusion
This EKP prototype proves that LEGO can move from **reactive data management** to **proactive knowledge orchestration**.

## 10. Future Horizon: EKP for Operational Excellence (LEAN / Six Sigma)
**Why this matters**: LEGO is a world leader in **Continuous Improvement**. While out of scope for v1.0, the EKP architecture is designed to support **Operational Excellence** through these future hypotheses:

| Methodology | Hypothesis (Out of Scope) | Reasoning |
| :--- | :--- | :--- |
| **LEAN** | Eliminating "Information Muda" (Waste) in data-cleansing. | Planners spend 70% of their time on data prep; EKP automates this. |
| **Six Sigma** | Reducing "Decision Variance" through a single semantic truth. | Ensuring Logistics and Finance never have conflicting data reports. |
| **JIT / JIC** | Just-In-Time Resilience via real-time risk propagation. | Reducing the cost of "Just-In-Case" overstocking of high-risk parts. |

## 11. Advanced SCM: PSI Control Tower & Ripple Effect Heatmap (T-11)

In this section, we demonstrate the **Orchestration** of supply and demand. By synchronizing **Purchases, Sales, and Inventory (PSI)**, we can identify potential stock-outs before they happen.

**Key Concepts:**
- **PSI Balance**: Monitoring if $Inventory + Purchases$ can meet $Forecasted Demand$.
- **Ripple Effect ($\alpha=1.2$)**: Modeling how a 15-day delay in Tier 2 material (e.g., ABS resin) amplifies into a 21.6-day delay for the finished set.
- **CPFR (Collaborative Planning)**: Tagging suppliers who share real-time forecasts to reduce the "Information Bullwhip."

**Executive Summary**: The 'Control Tower' view allows S&OP leaders to see regional imbalances in seconds, while the 'Ripple Effect' simulation predicts downstream delays before the Tier 1 suppliers even report them.

In [ ]:
# 1. PSI Control Tower: Supply-Demand Gap Analysis
psi_query = """
MATCH (f:Factory)-[p:PLANS_FOR]->(reg:Region)-[d:FORECASTS_DEMAND]->(s:Set)
WITH reg, f, p, sum(d.volume) AS total_demand
RETURN reg.name AS Region, 
       f.name AS Primary_Factory, 
       p.inventory_level AS Current_Inventory, 
       p.safety_stock AS Safety_Stock,
       round(total_demand) AS Forecasted_Demand,
       CASE WHEN (p.inventory_level < total_demand) THEN 'STOCK-OUT RISK' ELSE 'HEALTHY' END AS Status
"""
try:
    psi_df = run_query(psi_query)
    print("--- PSI Control Tower: Regional Supply-Demand Balance ---")
    print(psi_df)
except Exception as e:
    print(f"PSI Query Failed: {e}")

In [ ]:
# 2. Ripple Effect Heatmap: Lead-Time Amplification (Alpha=1.2)
ripple_query = """
MATCH (s2:Supplier {location: 'Taiwan'})-[r1:SOURCES_TO]->(s1:Supplier)-[r2:SOURCES_TO]->(f:Factory)-[r3:TRANSFORMS_TO]->(set:Set)
WHERE r1.current_delay > 0
WITH set, f, r1,
     (r1.current_delay * 1.44) AS Propagated_Delay
RETURN set.name AS Set_Name, 
       f.name AS Factory, 
       round(Propagated_Delay, 1) AS Days_Delayed,
       'High' AS Impact_Severity
ORDER BY Propagated_Delay DESC
LIMIT 10
"""
try:
    ripple_df = run_query(ripple_query)
    print("--- Ripple Effect: Tier 2 Disruption Propagation (Taiwan ABS Resin) ---")
    print(ripple_df)
except Exception as e:
    print(f"Ripple Query Failed: {e}")

In [ ]:
# 3. CPFR: Collaborative Planning Visibility
cpfr_query = """
MATCH (s:Supplier)-[r:SOURCES_TO]->(f:Factory)
WHERE r.is_collaborative = true
RETURN s.name AS Supplier, f.name AS Factory, round(r.inbound_volume) AS Inbound_Vol, 'Collaborative' AS Workflow
LIMIT 10
"""
try:
    cpfr_df = run_query(cpfr_query)
    print("--- CPFR: Active Collaborative Sourcing Partners ---")
    print(cpfr_df)
except Exception as e:
    print(f"CPFR Query Failed: {e}")

## 12. Executive Digital Twin Control Tower: Strategic Portfolio Maturity

This final section represents the "Value Realization" phase for the Senior PM. We aggregate technical data into an **EKP Maturity Scorecard** for executive decision-making.

**Strategic Insight**: By combining **Supply Resilience** and **Sustainability (DtT)**, we can identify which product lines are truly "Future-Proof."

**Executive Summary**: The scorecard below identifies 'Portfolio Champions' (High Resilience, Low DtT) and 'Strategic Risks' that require urgent R&D or logistics intervention.

In [ ]:
# 1. Portfolio Maturity Scorecard
maturity_query = """
MATCH (t:Theme)
WHERE t.distance_to_2032_target IS NOT NULL
OPTIONAL MATCH (t)<-[:IN_THEME]-(s:Set)
WITH t, count(s) AS total_sets, 
     t.sustainability_completion_pct AS sc_pct, 
     t.distance_to_2032_target AS dtt_pct
RETURN t.name AS Theme, 
       total_sets AS SKU_Count,
       round(sc_pct, 1) AS ESG_Maturity_Pct,
       round(dtt_pct, 1) AS Strategic_Gap_Pct,
       CASE 
           WHEN sc_pct > 70 THEN '🏆 CHAMPION' 
           WHEN sc_pct < 30 THEN '⚠️ RISK' 
           ELSE '🔄 IN-PROGRESS' 
       END AS Executive_Status
ORDER BY ESG_Maturity_Pct DESC
"""
try:
    maturity_df = run_query(maturity_query)
    print("--- EKP Strategic Maturity Scorecard ---")
    print(maturity_df)
except Exception as e:
    print(f"Maturity Scorecard Failed: {e}")

## 13. AI Ethics & Synthetic Data Disclaimer
This prototype utilizes a hybrid of **Real** (Rebrickable, official LEGO Supplier lists) and **Synthetic** (simulated Tier 2 relationships) data. 

**Transparency & Diligence**:
- Every node in the graph contains a `source` and `is_synthetic` property.
- Simulated relationships are intended for **Scenario Modeling** only and do not reflect real-world LEGO proprietary supply chain configurations.
- This approach ensures that we can demonstrate **Platform Potential** (e.g., Tier 2 Risk) while strictly adhering to data ethics and avoiding the publication of sensitive information.

*Built with Gemini CLI for the LEGO Enterprise Knowledge Platform.*